In [1]:
from pathlib import Path
import hashlib
import pandas as pd
import json
import sys

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.paths import Paths
from src.utils.checksums import calculate_sha256

PROJECT_ROOT = Paths.root
RAW_DIR = Paths.root / "data" / "raw"
AUDIT_DIR = Paths.audit
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
REGISTRY_PATH = PROJECT_ROOT / "config" / "raw_data_registry.json"


In [2]:
# Load the raw data registry
with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
    RAW_DATA_REGISTRY = json.load(f)

RAW_DATA_REGISTRY.keys()

dict_keys(['depmap', 'gdsc', 'ctrp', 'prism', 'tcga', 'lincs', 'cell_model_passports', 'chembl', 'drugbank'])

In [3]:
# Iterate through raw data files and collect metadata for the audit
records = []
missing_metadata = []

for file_path in RAW_DIR.rglob("*"):

    if not file_path.is_file():
        continue

    # Skip repository structure files
    if file_path.name == ".gitkeep":
        continue

    dataset = file_path.parent.name
    file_name = file_path.name

    dataset_metadata = RAW_DATA_REGISTRY.get(dataset)

    if dataset_metadata is None:
        missing_metadata.append(str(file_path.relative_to(PROJECT_ROOT)))
        continue

    file_metadata = dataset_metadata.get("files", {}).get(file_name)

    if file_metadata is None:
        missing_metadata.append(str(file_path.relative_to(PROJECT_ROOT)))
        continue

    records.append(
        {
            "relative_path": str(file_path.relative_to(PROJECT_ROOT)).replace("\\", "/"),
            "file_name": file_name,
            "dataset": dataset,
            "source_database": dataset_metadata.get("source_database"),
            "provider": dataset_metadata.get("provider"),
            "release": dataset_metadata.get("release"),
            "download_page_url": dataset_metadata.get("download_page_url"),
            "file_url": file_metadata.get("file_url"),
            "size_bytes": file_path.stat().st_size,
            "sha256": calculate_sha256(file_path),
        }
    )

# Create a DataFrame from the records and save it as a JSON file
if missing_metadata:
    print("WARNING: Missing metadata for:")
    for item in missing_metadata:
        print(f"  - {item}")


# Create a DataFrame from the records and save it as a JSON file
raw_file_audit = (
    pd.DataFrame(records)
    .sort_values(["dataset", "file_name"])
    .reset_index(drop=True)
)

# Save the audit DataFrame to a JSON file
audit_records = (
    raw_file_audit
    .astype(object)
    .where(pd.notna(raw_file_audit), None)
    .to_dict(orient="records")
)

output_path = AUDIT_DIR / "raw_file_audit.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        audit_records,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False
    )

# Validate JSON output
with open(output_path, "r", encoding="utf-8") as f:
    json.load(f)

print(f"Audit saved to: {output_path}")
print(f"Audited files: {len(raw_file_audit)}")
print("JSON validation passed.")

raw_file_audit

Audit saved to: C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\data\interim\qc\raw_file_audit.json
Audited files: 9
JSON validation passed.


,relative_path,file_name,dataset,source_database,provider,release,download_page_url,file_url,size_bytes,sha256
0,data/raw/depmap/Model.csv,Model.csv,depmap,DepMap,None,Public 24Q4,https://depmap.org/portal/download/,https://depmap.org/portal/data_page/?tab=allDa...,645696,b7a0c1385e6cef30132b56aff61f1261d11e3f490490b3...
1,data/raw/depmap/OmicsDefaultModelProfiles.csv,OmicsDefaultModelProfiles.csv,depmap,DepMap,None,Public 24Q4,https://depmap.org/portal/download/,https://depmap.org/portal/data_page/?tab=allDa...,90080,096a96c39d88374cb7c37058816e6cde29d6617b820b8a...
2,data/raw/depmap/OmicsExpressionProteinCodingGe...,OmicsExpressionProteinCodingGenesTPMLogp1.csv,depmap,DepMap,None,Public 24Q4,https://depmap.org/portal/download/,https://depmap.org/portal/data_page/?tab=allDa...,506628654,2a71dc94110efcc0221eae821bb93a9f03b54bea16f005...
3,data/raw/depmap/OmicsProfiles.csv,OmicsProfiles.csv,depmap,DepMap,None,Public 24Q4,https://depmap.org/portal/download/,https://depmap.org/portal/data_page/?tab=allDa...,254733,0bb19ee57935d4b213d378a4db9caacaa1db7318a63ae8...
4,data/raw/depmap/OmicsSomaticMutations.csv,OmicsSomaticMutations.csv,depmap,DepMap,None,Public 24Q4,https://depmap.org/portal/download/,https://depmap.org/portal/data_page/?tab=allDa...,338945382,92e86ab6a8457ef99908daf2a2c82d9529320e23cd6c6b...
5,data/raw/gdsc/GDSC2_fitted_dose_response_27Oct...,GDSC2_fitted_dose_response_27Oct23.xlsx,gdsc,Genomics of Drug Sensitivity in Cancer (GDSC),Cell Model Passports,GDSC2,https://cellmodelpassports.sanger.ac.uk/downloads,https://cmp.cog.sanger.ac.uk/download/GDSC2_fi...,22112537,0e99120a14a003dcbd3c27c020f1cfd35e38ee091067a2...
6,data/raw/tcga/gdc_manifest_tcga_primary_tumor_...,gdc_manifest_tcga_primary_tumor_rnaseq_star_co...,tcga,Genomic Data Commons (GDC),National Cancer Institute,"GDC Data Portal query export, 2026-06-12",https://portal.gdc.cancer.gov/repository,GDC Data Portal manifest generated from query ...,1680231,3fcbb6353753d47a851826005500ca6de7f58a37344790...
7,data/raw/tcga/gdc_metadata_tcga_primary_tumor_...,gdc_metadata_tcga_primary_tumor_rnaseq_star_co...,tcga,Genomic Data Commons (GDC),National Cancer Institute,"GDC Data Portal query export, 2026-06-12",https://portal.gdc.cancer.gov/repository,GDC Data Portal metadata export generated from...,27073623,7656b10f8d45fad4d33af9fccfd50ea48884b76569dc9f...
8,data/raw/tcga/gdc_sample_sheet_tcga_primary_tu...,gdc_sample_sheet_tcga_primary_tumor_rnaseq_sta...,tcga,Genomic Data Commons (GDC),National Cancer Institute,"GDC Data Portal query export, 2026-06-12",https://portal.gdc.cancer.gov/repository,GDC Data Portal sample sheet generated from qu...,2471521,bcb922f064d2aa4c60c4f10f1afd0ebe6db657a8d7740e...
